# Практика 2. MLflow: журнал экспериментов и реестр моделей

**Перед началом:** этот файл только для чтения и общий для всей группы.
Сохраните свою копию: **File → Save Notebook As…** → `work/students/<ваше_имя>/02_mlflow.ipynb`.

Веб-интерфейс MLflow: **http://10.162.1.86:5000** — войдите своим логином, держите вкладку открытой
и смотрите, как после каждой ячейки там появляются запуски.

План:
1. Установка
2. Подключение со своим логином
3. **Первые шаги: от простого к сложному** — один запуск, несколько подряд, ветвление на два параллельных
4. Главные понятия
5. Эксперимент на настоящих данных
6. Артефакты: графики, отчёты, файлы
7. Сохранение и загрузка модели
8. Перебор параметров и поиск лучшего запуска
9. Метрики по эпохам (как в deep learning)
10. Реестр моделей: версии и метка `champion`
11. Автологирование
12. **Задачи**

## 1. Установка

- **mlflow-skinny** — лёгкий клиент MLflow (без веб-сервера: сервер у нас уже запущен). Версия клиента должна
  совпадать с версией сервера — у нас **3.16.1**.
- **skops** — безопасный формат сохранения моделей sklearn, его использует MLflow 3.

На сервере всё установлено, команда только проверит.

In [ ]:
%pip install -q "mlflow-skinny==3.16.1" skops scikit-learn matplotlib

In [ ]:
import mlflow
print("mlflow", mlflow.__version__)

## 2. Подключение

| Что | Значение |
|---|---|
| адрес сервера (`MLFLOW_TRACKING_URI`) | внутри сервера `http://mlflow:5000`, снаружи `http://10.162.1.86:5000` |
| логин (`MLFLOW_TRACKING_USERNAME`) | ваш логин: `user1` … `user30` |
| пароль (`MLFLOW_TRACKING_PASSWORD`) | выдаёт преподаватель |

MLflow берёт адрес, логин и пароль **из переменных окружения** — так устроен клиент MLflow:

| Переменная | Что в ней |
|---|---|
| `MLFLOW_TRACKING_URI` | адрес сервера MLflow |
| `MLFLOW_TRACKING_USERNAME` | логин |
| `MLFLOW_TRACKING_PASSWORD` | пароль |

Пароль вводим через `getpass` — в коде его нет.

> Работаете **на своём компьютере** (VS Code, Anaconda)? Адрес `http://mlflow:5000` там не работает —
> это внутреннее имя сервера. Ячейка ниже сама возьмёт внешний адрес `http://10.162.1.86:5000`.

**Права:** вы видите эксперименты всей группы, но менять можете только **свои**.
Поэтому все имена начинаем со своего логина: `user7_wine`, `user7_wine_model`.

In [ ]:
import os
from getpass import getpass

MLFLOW_USER = "user1"   # <-- ЗАМЕНИТЕ на свой логин: user1 … user30

# адрес: в Jupyter на сервере уже задан (http://mlflow:5000), на своём компьютере — внешний
os.environ.setdefault("MLFLOW_TRACKING_URI", "http://10.162.1.86:5000")
os.environ["MLFLOW_TRACKING_USERNAME"] = MLFLOW_USER
os.environ["MLFLOW_TRACKING_PASSWORD"] = os.environ.get("NB_MLFLOW_PASSWORD") or getpass(f"Пароль MLflow для {MLFLOW_USER}: ")

UI = "http://10.162.1.86:5000"   # адрес интерфейса для ссылок в браузере
print("сервер MLflow:", os.environ["MLFLOW_TRACKING_URI"], "| логин:", MLFLOW_USER)

In [ ]:
# проверка связи: 5 последних экспериментов на сервере
for e in mlflow.search_experiments(max_results=5, order_by=["last_update_time DESC"]):
    print(e.experiment_id, e.name)

## 3. Первые шаги: от простого к сложному

Прежде чем обучать модели, посмотрим, как MLflow записывает запуски, на самых простых примерах.
Держите открытой вкладку **http://10.162.1.86:5000** и после каждой ячейки обновляйте страницу.

### 3.1 Один запуск
Минимум, который нужен MLflow: **эксперимент** (папка) и **запуск** внутри него.

In [ ]:
mlflow.set_experiment(f"{MLFLOW_USER}_first_steps")       # папка для запусков (создастся сама)

with mlflow.start_run(run_name="step1_hello"):             # один запуск
    mlflow.log_param("learning_rate", 0.1)                 # параметр: с чем запускали
    mlflow.log_metric("accuracy", 0.85)                    # метрика: что получилось

print("готово — в интерфейсе появился эксперимент", f"{MLFLOW_USER}_first_steps", "и в нём 1 запуск")

### 3.2 Несколько запусков подряд
Каждая итерация цикла — **отдельный запуск**. Так делают, когда перебирают параметры:
в интерфейсе запуски встанут строками таблицы, и их можно сравнить.

In [ ]:
for lr in [0.001, 0.01, 0.1]:
    with mlflow.start_run(run_name=f"step2_lr_{lr}"):
        mlflow.log_param("learning_rate", lr)
        mlflow.log_metric("accuracy", round(0.7 + lr, 3))  # «результат» — пока просто число
        print("запуск с learning_rate =", lr)

runs = mlflow.search_runs(experiment_names=[f"{MLFLOW_USER}_first_steps"], order_by=["start_time ASC"])
runs[["tags.mlflow.runName", "params.learning_rate", "metrics.accuracy"]]

### 3.3 Один запуск разветвляется на два
Часто один шаг готовит данные, а дальше из них обучают **несколько моделей**.
В MLflow это **родительский запуск** с **дочерними** (`nested=True`):

```
pipeline (родитель: данные)
 ├── logreg          (дочерний)
 └── decision_tree   (дочерний)
```
В интерфейсе дочерние запуски спрятаны под родителем — нажмите «+» слева от `pipeline`.

In [ ]:
from sklearn.datasets import load_wine
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import f1_score

X, y = load_wine(return_X_y=True, as_frame=True)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=42, stratify=y)

with mlflow.start_run(run_name="step3_pipeline"):                          # РОДИТЕЛЬ: данные
    mlflow.log_params({"dataset": "wine", "train_rows": len(X_train), "test_rows": len(X_test)})

    with mlflow.start_run(run_name="logreg", nested=True):                 # ребёнок 1
        m = make_pipeline(StandardScaler(), LogisticRegression(max_iter=5000)).fit(X_train, y_train)
        mlflow.log_metric("f1_macro", f1_score(y_test, m.predict(X_test), average="macro"))

    with mlflow.start_run(run_name="decision_tree", nested=True):          # ребёнок 2
        m = DecisionTreeClassifier(max_depth=2, random_state=42).fit(X_train, y_train)
        mlflow.log_metric("f1_macro", f1_score(y_test, m.predict(X_test), average="macro"))

print("родитель и 2 дочерних запуска записаны")

### 3.4 Два дочерних запуска параллельно, результат — обратно в родителя
В 3.3 модели обучались **по очереди**. Если модели обучаются долго (нейросети — часами),
их запускают **одновременно**. После этого родитель собирает результаты и записывает лучшую модель:

```
                ┌──► logreg          ─┐
step4_pipeline ─┤                     ├──► лучшая модель → в родителя
                └──► decision_tree   ─┘
```
Каждый поток открывает свой дочерний запуск и явно указывает родителя (`parent_run_id`).

In [ ]:
import time
from concurrent.futures import ThreadPoolExecutor

def train_child(name, model, parent_id):
    """Обучает одну модель в своём дочернем запуске. Возвращает (имя, f1, время старта, время конца)."""
    with mlflow.start_run(run_name=name, parent_run_id=parent_id, nested=True):
        start = time.time()
        model.fit(X_train, y_train)
        time.sleep(2)                                       # имитация долгого обучения
        f1 = f1_score(y_test, model.predict(X_test), average="macro")
        mlflow.log_metric("f1_macro", f1)
        return name, f1, start, time.time()

models = {
    "logreg": make_pipeline(StandardScaler(), LogisticRegression(max_iter=5000)),
    "decision_tree": DecisionTreeClassifier(max_depth=2, random_state=42),
}

with mlflow.start_run(run_name="step4_parallel") as parent:
    mlflow.log_param("dataset", "wine")
    t0 = time.time()
    with ThreadPoolExecutor(max_workers=2) as pool:                        # два потока одновременно
        results = list(pool.map(lambda item: train_child(*item, parent.info.run_id), models.items()))

    best_name, best_f1, _, _ = max(results, key=lambda r: r[1])           # сбор результатов
    mlflow.log_params({"best_model": best_name})
    mlflow.log_metric("best_f1", best_f1)

for name, f1, start, end in results:
    print(f"{name:<14} f1={f1:.3f}   обучение с {start - t0:.1f} по {end - t0:.1f} с")
parallel = max(r[3] for r in results) - min(r[2] for r in results)     # от первого старта до последнего конца
one_by_one = sum(r[3] - r[2] for r in results)                          # если бы обучали по очереди
print(f"обучение заняло {parallel:.1f} с, по очереди было бы {one_by_one:.1f} с | лучшая: {best_name}")

Посмотрите: обе модели начали и закончили **в одно время** — они шли параллельно.
Это та же схема, по которой в Airflow одна задача раздаёт работу нескольким параллельным задачам,
а следующая задача собирает результаты. Дальше разберём всё подробно на настоящих моделях.

## 4. Главные понятия

| Понятие | Что это | Пример |
|---|---|---|
| **Experiment** | папка для запусков одной задачи | `user7_wine` |
| **Run** | один запуск обучения | «логрегрессия, C=1» |
| **Params** | с чем обучали (входы) | `C=1.0`, `max_iter=5000` |
| **Metrics** | что получилось (числа, можно по шагам) | `accuracy=0.97` |
| **Tags** | подписи для поиска | `model_type=logreg` |
| **Artifacts** | файлы запуска | график, отчёт, модель |
| **Registered model** | модель в реестре с версиями | `user7_wine_model` v1, v2… |
| **Alias** | метка версии | `@champion` — основная |

## 5. Эксперимент на настоящих данных

`set_experiment` выбирает эксперимент (и создаёт, если его нет). Всё, что внутри `with mlflow.start_run():`,
попадает в один запуск.

> После каждого запуска MLflow сам печатает строку `🏃 View run … at: http://mlflow:5000/…`.
> `mlflow` — внутреннее имя сервера, из браузера оно не откроется. Открывайте ссылки, которые печатаем мы
> (с адресом `http://10.162.1.86:5000`), или замените в ссылке `mlflow` на `10.162.1.86`.

In [ ]:
import pandas as pd
from sklearn.datasets import load_wine
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, f1_score

X, y = load_wine(return_X_y=True, as_frame=True)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=42, stratify=y)

EXPERIMENT = f"{MLFLOW_USER}_wine"
exp = mlflow.set_experiment(EXPERIMENT)
print("эксперимент:", exp.name, "id =", exp.experiment_id)

In [ ]:
params = {"C": 1.0, "max_iter": 5000}

with mlflow.start_run(run_name="logreg_C1") as run:
    mlflow.log_params(params)                                   # 1) с чем обучали
    mlflow.set_tag("model_type", "logreg")                      # 2) подпись для поиска

    model = make_pipeline(StandardScaler(), LogisticRegression(**params)).fit(X_train, y_train)
    pred = model.predict(X_test)

    mlflow.log_metrics({                                        # 3) что получилось
        "accuracy": accuracy_score(y_test, pred),
        "f1_macro": f1_score(y_test, pred, average="macro"),
    })

print("run id:", run.info.run_id)
print("открыть в браузере:", f"{UI}/#/experiments/{exp.experiment_id}/runs/{run.info.run_id}")

## 6. Артефакты

Артефакт — любой файл запуска. Файлы MLflow сам кладёт в S3 (бакет `mlflow`), вам ключи S3 не нужны.

| Команда | Что сохраняет |
|---|---|
| `mlflow.log_figure(fig, "cm.png")` | график matplotlib |
| `mlflow.log_dict(d, "report.json")` | словарь как JSON |
| `mlflow.log_text(s, "notes.txt")` | строку как текстовый файл |
| `mlflow.log_artifact("путь")` | любой файл с диска |

In [ ]:
import matplotlib.pyplot as plt
from sklearn.metrics import ConfusionMatrixDisplay, classification_report

with mlflow.start_run(run_name="logreg_with_artifacts") as run:
    mlflow.log_params(params)
    model = make_pipeline(StandardScaler(), LogisticRegression(**params)).fit(X_train, y_train)
    pred = model.predict(X_test)
    mlflow.log_metric("accuracy", accuracy_score(y_test, pred))

    disp = ConfusionMatrixDisplay.from_predictions(y_test, pred, cmap="Blues")
    mlflow.log_figure(disp.figure_, "confusion_matrix.png")      # график
    plt.close(disp.figure_)

    mlflow.log_dict(classification_report(y_test, pred, output_dict=True), "report.json")   # отчёт
    mlflow.log_text("Базовая модель для сравнения. Данные: sklearn wine.", "notes.txt")      # заметка

client = mlflow.MlflowClient()
print([a.path for a in client.list_artifacts(run.info.run_id)])

## 7. Сохранение и загрузка модели

`mlflow.sklearn.log_model` сохраняет модель в запуск вместе с описанием входа (`signature`) и примером данных.
MLflow 3 пишет модели sklearn в безопасном формате **skops**, и ему нужно заранее сказать, какие внутренние типы модели
считать безопасными — функция ниже считает их автоматически. Копируйте её как есть.

In [ ]:
import skops.io as sio
from mlflow.models import infer_signature

def log_sklearn_model(model, X_example):
    """Сохраняет sklearn-модель в текущий запуск MLflow. Возвращает ModelInfo (в нём model_uri)."""
    return mlflow.sklearn.log_model(
        model,
        name="model",
        signature=infer_signature(X_example, model.predict(X_example)),
        input_example=X_example.head(3),
        skops_trusted_types=sio.get_untrusted_types(data=sio.dumps(model)),
    )

with mlflow.start_run(run_name="logreg_with_model") as run:
    mlflow.log_params(params)
    model = make_pipeline(StandardScaler(), LogisticRegression(**params)).fit(X_train, y_train)
    mlflow.log_metric("accuracy", accuracy_score(y_test, model.predict(X_test)))
    info = log_sklearn_model(model, X_test)

print("model_uri:", info.model_uri)

In [ ]:
# загрузить модель обратно — на любом компьютере, где есть доступ к MLflow
loaded = mlflow.sklearn.load_model(info.model_uri)
print("прогноз загруженной модели:", loaded.predict(X_test.head(5)), "| правильно:", y_test.head(5).tolist())

## 8. Перебор параметров и поиск лучшего

Один параметр — один запуск. Потом `search_runs` возвращает **все запуски таблицей pandas**:
их можно сортировать и фильтровать, как обычный DataFrame.

In [ ]:
from sklearn.ensemble import RandomForestClassifier

for n in [5, 20, 100]:
    with mlflow.start_run(run_name=f"rf_{n}"):
        mlflow.set_tag("model_type", "random_forest")
        mlflow.log_param("n_estimators", n)
        rf = RandomForestClassifier(n_estimators=n, random_state=42).fit(X_train, y_train)
        mlflow.log_metric("f1_macro", f1_score(y_test, rf.predict(X_test), average="macro"))
        log_sklearn_model(rf, X_test)
print("готово")

In [ ]:
runs = mlflow.search_runs(
    experiment_names=[EXPERIMENT],
    filter_string="tags.model_type = 'random_forest'",
    order_by=["metrics.f1_macro DESC"],
)
runs[["run_id", "tags.mlflow.runName", "params.n_estimators", "metrics.f1_macro"]]

In [ ]:
best = runs.iloc[0]
print("лучший запуск:", best["tags.mlflow.runName"], "f1 =", round(best["metrics.f1_macro"], 3))

**В интерфейсе:** откройте эксперимент, отметьте галочками три запуска `rf_*` и нажмите **Compare** —
увидите параметры и метрики рядом и график зависимости f1 от числа деревьев.

## 9. Метрики по эпохам — как в deep learning

Нейросеть учится эпохами, и после каждой эпохи мы записываем метрику с номером шага `step`.
MLflow сам строит из этого **кривую обучения**. Покажем на `SGDClassifier`, который тоже учится по эпохам (`partial_fit`).

In [ ]:
import numpy as np
from sklearn.linear_model import SGDClassifier

scaler = StandardScaler().fit(X_train)
Xtr, Xte = scaler.transform(X_train), scaler.transform(X_test)
classes = np.unique(y_train)

with mlflow.start_run(run_name="sgd_epochs") as run:
    mlflow.log_params({"model": "SGDClassifier", "epochs": 30, "learning_rate": "optimal"})
    sgd = SGDClassifier(loss="log_loss", random_state=42)
    for epoch in range(30):
        sgd.partial_fit(Xtr, y_train, classes=classes)                      # одна эпоха
        mlflow.log_metrics({
            "train_accuracy": sgd.score(Xtr, y_train),
            "val_accuracy": sgd.score(Xte, y_test),
        }, step=epoch)                                                      # <- номер эпохи

history = mlflow.MlflowClient().get_metric_history(run.info.run_id, "val_accuracy")
print("val_accuracy по эпохам:", [round(m.value, 2) for m in history[:10]], "...")
print("график:", f"{UI}/#/experiments/{exp.experiment_id}/runs/{run.info.run_id}")

## 10. Реестр моделей: версии и метка `champion`

- `mlflow.register_model(uri, имя)` — сделать модель новой **версией** в реестре;
- `set_registered_model_alias(имя, "champion", версия)` — отметить основную версию;
- `load_model("models:/имя@champion")` — загрузить основную версию, **не зная её номера**.

Откатиться на старую версию = переставить метку. Код прогноза при этом не меняется.

In [ ]:
MODEL_NAME = f"{MLFLOW_USER}_wine_model"
client = mlflow.MlflowClient()

version = mlflow.register_model(f"runs:/{best['run_id']}/model", MODEL_NAME)
client.set_registered_model_alias(MODEL_NAME, "champion", version.version)
print(f"{MODEL_NAME}: версия {version.version} получила метку @champion")

In [ ]:
champion = mlflow.sklearn.load_model(f"models:/{MODEL_NAME}@champion")
print("прогноз чемпиона:", champion.predict(X_test.head(5)))

In [ ]:
# все версии модели и их метки
for v in client.search_model_versions(f"name='{MODEL_NAME}'"):
    mv = client.get_model_version(MODEL_NAME, v.version)          # здесь есть метки (aliases)
    print("версия", mv.version, "| метки:", mv.aliases, "| из запуска", mv.run_id[:8])

## 11. Автологирование

`mlflow.sklearn.autolog()` — MLflow сам записывает параметры, метрики на обучающей выборке и модель
при каждом `fit`. Удобно для быстрых экспериментов. Для PyTorch и Keras есть `mlflow.pytorch.autolog()`
и `mlflow.keras.autolog()`.

In [ ]:
mlflow.sklearn.autolog(log_models=False)          # модель не сохраняем, только параметры и метрики

with mlflow.start_run(run_name="autolog_logreg") as run:
    make_pipeline(StandardScaler(), LogisticRegression(C=0.5, max_iter=5000)).fit(X_train, y_train)

mlflow.sklearn.autolog(disable=True)              # выключаем, чтобы не мешало дальше
r = client.get_run(run.info.run_id)
print("параметров записано сами:", len(r.data.params), "| метрики:", list(r.data.metrics)[:4])

## Шпаргалка

| Действие | Команда |
|---|---|
| эксперимент | `mlflow.set_experiment(f"{MLFLOW_USER}_...")` |
| запуск | `with mlflow.start_run(run_name="..."):` |
| параметры / метрики | `mlflow.log_params({...})`, `mlflow.log_metrics({...}, step=epoch)` |
| подпись | `mlflow.set_tag("model_type", "rf")` |
| артефакты | `log_figure`, `log_dict`, `log_text`, `log_artifact` |
| модель | `mlflow.sklearn.log_model(...)` → `mlflow.sklearn.load_model(uri)` |
| все запуски | `mlflow.search_runs(experiment_names=[...], order_by=[...])` |
| история метрики | `MlflowClient().get_metric_history(run_id, "val_accuracy")` |
| реестр | `mlflow.register_model(uri, name)`, `set_registered_model_alias(name, "champion", v)` |
| загрузить основную | `mlflow.sklearn.load_model(f"models:/{name}@champion")` |

---
# 12. Задачи

Датасет для всех задач — **Breast Cancer** (`sklearn.datasets.load_breast_cancer`): по признакам клеток
определить, злокачественная опухоль или нет. Эксперимент — `f"{MLFLOW_USER}_cancer"`.
После каждой задачи проверяйте результат в интерфейсе MLflow.

In [ ]:
# данные для задач — запустите эту ячейку
from sklearn.datasets import load_breast_cancer

Xc, yc = load_breast_cancer(return_X_y=True, as_frame=True)
Xc_train, Xc_test, yc_train, yc_test = train_test_split(Xc, yc, test_size=0.25, random_state=42, stratify=yc)
CANCER_EXP = f"{MLFLOW_USER}_cancer"
print(Xc.shape, "| эксперимент:", CANCER_EXP)

### Задача 1. Первый запуск
В эксперименте `CANCER_EXP` сделайте запуск `logreg`: `LogisticRegression(max_iter=5000)` со стандартизацией.
Запишите параметр `max_iter` и три метрики на тесте: `accuracy`, `f1`, `roc_auc`
(для ROC-AUC нужен `predict_proba(...)[:, 1]`).

In [ ]:
# ваш код здесь

### Задача 2. Сравнение трёх моделей
Обучите три модели, каждую — отдельным запуском с тегом `model_type`: логистическая регрессия,
`RandomForestClassifier(random_state=42)`, `KNeighborsClassifier()` (две последние тоже со стандартизацией).
Запишите `roc_auc` и сохраните модель (`log_sklearn_model`). Через `search_runs` выведите таблицу,
отсортированную по `roc_auc`, и напечатайте лучшую модель.

In [ ]:
# ваш код здесь

### Задача 3. Графики лучшей модели
Загрузите лучшую модель из задачи 2 (`runs:/<run_id>/model`) и **в тот же запуск** добавьте
матрицу ошибок и ROC-кривую (`RocCurveDisplay.from_predictions`) как артефакты.
Подсказка: продолжить существующий запуск — `mlflow.start_run(run_id=...)`.

In [ ]:
# ваш код здесь

### Задача 4. Кривая обучения
Обучите `SGDClassifier(loss="log_loss", random_state=42)` на Breast Cancer 40 эпохами (`partial_fit`)
и после каждой эпохи записывайте `train_accuracy` и `val_accuracy` со `step=epoch`.
Потом через `get_metric_history` найдите эпоху с лучшим `val_accuracy`. Посмотрите график в интерфейсе.

In [ ]:
# ваш код здесь

### Задача 5. Реестр
Зарегистрируйте лучшую модель из задачи 2 как `f"{MLFLOW_USER}_cancer_model"`, поставьте ей метку `champion`,
загрузите по адресу `models:/…@champion` и сделайте прогноз для первых 5 строк теста.

In [ ]:
# ваш код здесь

### Задача 6 ★. Ворота качества
Обучите новую модель — `RandomForestClassifier(n_estimators=500, max_depth=5, random_state=42)`.
Узнайте `roc_auc` текущего чемпиона (метка → версия → запуск → метрика) и сравните.
Зарегистрируйте новую модель и переставьте `champion` на неё **только если она лучше**.
Напечатайте, кто в итоге чемпион. Так работает автоматический выпуск моделей в конвейере Airflow.

In [ ]:
# ваш код здесь

### Задача 7. Три модели параллельно под одним родителем
Сделайте родительский запуск `parallel_cancer` в `CANCER_EXP`. Внутри него **параллельно** (3 потока)
обучите три дочерние модели: логистическую регрессию, `RandomForestClassifier(random_state=42)`
и `KNeighborsClassifier()` (со стандартизацией). В каждой запишите `roc_auc`.
В родителя запишите `best_model` и `best_roc_auc`. Проверьте в интерфейсе, что под родителем три дочерних запуска.

In [ ]:
# ваш код здесь

### Задача 8. Интерфейс
Откройте **http://10.162.1.86:5000**, эксперимент `…_cancer`:
1. отметьте запуски `cmp_*` и нажмите **Compare** — какая модель лучше и насколько?
2. откройте запуск `sgd_40_epochs` → вкладка **Model metrics**: где начинается переобучение?
4. откройте **Models** → свою модель: сколько версий и у какой метка `champion`?

---
**Готово?** Следующий шаг — Airflow: тот же код запускается сам, по расписанию, и каждая задача DAG-а
пишет свой запуск в MLflow.